# ReSound · Separador de Pistas v3 (gratuito) · agora com Mapa de Batidas

Novidade desta versão: além das pistas de instrumentos, agora a **voz principal é separada dos backing vocals** automaticamente, usando um segundo modelo especializado em cima da pista de voz.

### Pistas que você recebe ao final
Voz principal · Backing vocals · Bateria · Baixo · Guitarra · Piano · Outros

### Como usar (passo a passo)
1. No menu acima: **Ambiente de execução > Alterar tipo de ambiente de execução**, escolha **T4 GPU** e salve.
2. Rode as células na ordem, clicando no play de cada uma.
3. Na célula 2, envie sua música quando o botão de upload aparecer.
4. Ao final, o ZIP com todas as pistas baixa sozinho. Descompacte e importe no **ReSound**.

*Tempo total para uma música de 3 a 4 minutos: por volta de 3 a 5 minutos com GPU (a primeira execução demora um pouco mais porque baixa os modelos).*

## Célula 1 · Instalar as ferramentas (rode uma vez por sessão)

In [ ]:
%%capture
!pip install -U demucs
!pip install "audio-separator[cpu]"
# A separacao de voz usa PyTorch, que aproveita a GPU sozinho.
# O onnxruntime em versao CPU evita conflitos de CUDA no Colab.


In [ ]:
print('Ferramentas instaladas! Pode seguir para a célula 2.')

## Célula 2 · Enviar sua música (MP3, WAV, M4A...)

In [ ]:
from google.colab import files
import os, shutil

if os.path.exists('/content/entrada'):
    shutil.rmtree('/content/entrada')
os.makedirs('/content/entrada', exist_ok=True)

print('Clique em "Escolher arquivos" e selecione sua música:')
enviados = files.upload()

for nome in enviados:
    shutil.move(nome, f'/content/entrada/{nome}')
    print(f'Recebido: {nome}')
arquivo = list(enviados.keys())[0]

## Célula 3 · Etapa 1: separar os instrumentos e a voz (Demucs)

- `htdemucs_6s` gera 6 pistas: voz (todas juntas), bateria, baixo, guitarra, piano e outros
- `htdemucs` gera 4 pistas: voz, bateria, baixo e outros (troque abaixo se preferir)

In [ ]:
MODO = 'htdemucs_6s'

import subprocess, os, shutil
if os.path.exists('/content/saida'):
    shutil.rmtree('/content/saida')
entrada = f'/content/entrada/{arquivo}'
print(f'Etapa 1: separando "{arquivo}" no modo {MODO}... aguarde.')
resultado = subprocess.run(
    ['demucs', '-n', MODO, '--mp3', '--mp3-bitrate', '320', '-o', '/content/saida', entrada],
    capture_output=True, text=True
)
if resultado.returncode == 0:
    print('Etapa 1 concluída! Instrumentos separados.')
else:
    print('Algo deu errado na etapa 1:')
    print((resultado.stderr or '')[-1500:])

## Célula 4 · Etapa 2: dividir a voz em PRINCIPAL e BACKING VOCALS

Esta etapa pega a pista de voz gerada acima e aplica um modelo especializado (tipo karaokê) que isola a voz principal dos vocais de apoio.

In [ ]:
import glob, os, shutil
from audio_separator.separator import Separator

# Localiza a pista de voz gerada pelo Demucs
candidatos = glob.glob('/content/saida/*/*/vocals.*')
if not candidatos:
    raise SystemExit('Pista de voz não encontrada. Rode a célula 3 primeiro.')
voz_completa = candidatos[0]
print('Pista de voz localizada:', voz_completa)

if os.path.exists('/content/vozes'):
    shutil.rmtree('/content/vozes')
os.makedirs('/content/vozes', exist_ok=True)

print('Etapa 2: separando voz principal dos backing vocals... aguarde.')
sep = Separator(output_dir='/content/vozes', output_format='mp3')
sep.load_model(model_filename='5_HP-Karaoke-UVR.pth')
saidas = sep.separate(voz_completa)

print('Etapa 2 concluída! Arquivos gerados:')
for s in saidas:
    print('  ·', os.path.basename(s))

## Célula 5 · Reunir tudo e baixar o ZIP final

In [ ]:
import shutil, os, glob
from google.colab import files

try:
    arquivo
except NameError:
    arquivo = os.path.basename(glob.glob('/content/entrada/*')[0])
base = os.path.splitext(arquivo)[0]
final = f'/content/{base}_pistas'
if os.path.exists(final):
    shutil.rmtree(final)
os.makedirs(final, exist_ok=True)

# 1) Instrumentos do Demucs (a voz completa fica de fora, pois foi dividida na etapa 2)
nomes_pt = {'drums':'Bateria','bass':'Baixo','other':'Outros','guitar':'Guitarra','piano':'Piano'}
pastas = [p for p in glob.glob('/content/saida/*/*') if os.path.isdir(p)]
if pastas:
    for f in glob.glob(f'{pastas[0]}/*'):
        raiz = os.path.splitext(os.path.basename(f))[0]
        ext = os.path.splitext(f)[1]
        if raiz == 'vocals':
            shutil.copy(f, f'{final}/Vozes juntas (referência) - {base}{ext}')
        elif raiz in nomes_pt:
            shutil.copy(f, f'{final}/{nomes_pt[raiz]} - {base}{ext}')

# 2) Voz principal e backing vocals da etapa 2
for f in glob.glob('/content/vozes/*'):
    nome = os.path.basename(f)
    ext = os.path.splitext(f)[1]
    if '(Vocals)' in nome:
        shutil.copy(f, f'{final}/Voz principal - {base}{ext}')
    elif '(Instrumental)' in nome:
        shutil.copy(f, f'{final}/Backing vocals - {base}{ext}')

zipado = shutil.make_archive(f'/content/{base}_pistas_ReSound', 'zip', final)
print('Pistas prontas para o ReSound:')
for f in sorted(os.listdir(final)):
    print('  ·', f)
files.download(zipado)
print('\nDownload iniciado! Descompacte e importe as pistas no ReSound.')


## Célula 6 · Mapa de Batidas profissional (opcional, recomendado para músicas sem bateria)

Gera um arquivo `_batidas.json` com **cada batida e cada tempo forte** da música, detectados por rede neural (a mesma família de tecnologia dos apps profissionais). Importe esse arquivo no metrônomo do ReSound e o clique seguirá a música com máxima precisão, mesmo em voz e violão, teclado ou andamento variável.

In [ ]:
import json, os, glob
import numpy as np
from google.colab import files

entrada_lista = glob.glob('/content/entrada/*')
if not entrada_lista:
    raise SystemExit('Envie a música na célula 2 primeiro.')
caminho = entrada_lista[0]
base = os.path.splitext(os.path.basename(caminho))[0]

beats, numeros, origem = None, None, None
try:
    import subprocess
    subprocess.run(['pip','install','-q','madmom'], check=True)
    from madmom.features.downbeats import RNNDownBeatProcessor, DBNDownBeatTrackingProcessor
    print('Analisando com rede neural (madmom)... 1 a 3 minutos.')
    act = RNNDownBeatProcessor()(caminho)
    proc = DBNDownBeatTrackingProcessor(beats_per_bar=[3,4], fps=100)
    res = proc(act)
    beats = res[:,0].tolist()
    numeros = [int(n) for n in res[:,1]]
    origem = 'madmom (rede neural)'
except Exception as e:
    print('madmom indisponível nesta sessão, usando o detector clássico (librosa)...')
    import librosa
    y, sr = librosa.load(caminho, sr=22050, mono=True)
    tempo, frames = librosa.beat.beat_track(y=y, sr=sr, trim=False)
    beats = librosa.frames_to_time(frames, sr=sr).tolist()
    env = librosa.onset.onset_strength(y=y, sr=sr)
    forca = [float(env[min(f, len(env)-1)]) for f in frames]
    melhor_o, melhor_s = 0, -1
    for o in range(4):
        s = sum(forca[o::4])
        if s > melhor_s: melhor_s, melhor_o = s, o
    numeros = [((i - melhor_o) % 4) + 1 for i in range(len(beats))]
    origem = 'librosa (clássico)'

iv = np.diff(beats)
bpm = round(float(60/np.median(iv)), 1) if len(iv) else 0
mapa = {'origem': origem, 'bpm': bpm, 'beats': [round(b,4) for b in beats], 'beat_numbers': numeros}
saida = f'/content/{base}_batidas.json'
json.dump(mapa, open(saida,'w'))
print(f'Mapa gerado: {len(beats)} batidas, ~{bpm} BPM, origem: {origem}')
files.download(saida)
print('Importe este arquivo no Metrônomo Inteligente do ReSound.')

---
### Dica de qualidade
Se os backing vocals saírem com resíduos da voz principal (ou vice-versa), é uma limitação natural desse tipo de separação, que depende muito de como as vozes foram mixadas. Vale testar também o modelo alternativo trocando, na célula 4, o nome do modelo por `UVR-BVE-4B_SN-44100-1.pth` e rodando as células 4 e 5 novamente. Cada música responde melhor a um modelo.

*ReSound · Separador v2.2 · uso gratuito via Google Colab*